# Housing Dataset Data Cleaning Workflow
This notebook demonstrates step-by-step data cleaning techniques applied to `Housing-selected-columns.csv`.

In [2]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
print("Libraries imported successfully.")

Libraries imported successfully.


## Step 1: Loading the Dataset
We load the housing dataset using pandas.

In [3]:
df=pd.read_csv('Housing-selected-columns.csv')

In [4]:
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning
0,13300000,7420,4,2,3,yes,no,no,no,yes
1,12250000,8960,4,4,4,yes,no,no,no,yes
2,12250000,9960,3,2,2,yes,no,yes,no,no
3,12215000,7500,4,2,2,yes,no,yes,no,yes
4,11410000,7420,4,1,2,yes,yes,yes,no,yes


## Step 2: Inspecting Data & Checking Missing/Duplicate Values

In [5]:
# Check structural details
print("--- Data Info ---")
df.info()


--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   price            545 non-null    int64 
 1   area             545 non-null    int64 
 2   bedrooms         545 non-null    int64 
 3   bathrooms        545 non-null    int64 
 4   stories          545 non-null    int64 
 5   mainroad         545 non-null    object
 6   guestroom        545 non-null    object
 7   basement         545 non-null    object
 8   hotwaterheating  545 non-null    object
 9   airconditioning  545 non-null    object
dtypes: int64(5), object(5)
memory usage: 42.7+ KB


In [6]:
# Check for missing values
print("\n--- Missing Values Count ---")
print(df.isnull().sum())


--- Missing Values Count ---
price              0
area               0
bedrooms           0
bathrooms          0
stories            0
mainroad           0
guestroom          0
basement           0
hotwaterheating    0
airconditioning    0
dtype: int64


In [7]:



# Check for duplicate rows
duplicate_count = df.duplicated().sum()
print(f"\nDuplicate rows found: {duplicate_count}")


Duplicate rows found: 0


## Step 3: Handling Missing Values (Imputation)
Though there are no missing values in this clean subset, we build a robust function to fill numerical nulls with the median and categorical ones with the mode.

In [8]:
# Make a working copy so the original df is preserved
df_clean = df.copy()

# Handle missing values 
# Numeric columns - fill with median (robust to outliers)
# Categorical columns - fill with mode (most frequent value)

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Missing values after handling:")
print(df_clean.isnull().sum().sum())

Missing values after handling:
0


## Step 4: Removing Duplicate Rows

In [9]:
duplicates_before = df_clean.duplicated().sum()
print(f"Duplicate rows found: {duplicates_before}")

df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df_clean.shape}")

Duplicate rows found: 0
Shape after removing duplicates: (545, 10)


## Step 5: Standardizing Categorical Columns (yes/no to binary 1/0)
We convert object/string values to standard lowercase and encode `yes`/`no` variables into numeric `1`/`0` tags.

In [11]:
df_standardized = df_clean.copy()
categorical_cols = df_standardized.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

for col in categorical_cols:
    # Trim whitespaces
    df_standardized[col] = df_standardized[col].astype(str).str.strip().str.lower()
    # Map 'yes'/'no' values
    if set(df_standardized[col].unique()).issubset({'yes', 'no'}):
        df_standardized[col] = df_standardized[col].map({'yes': 1, 'no': 0})
        print(f"Standardized '{col}' to binary 1/0")

df_standardized.head()

Categorical columns: ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning']
Standardized 'mainroad' to binary 1/0
Standardized 'guestroom' to binary 1/0
Standardized 'basement' to binary 1/0
Standardized 'hotwaterheating' to binary 1/0
Standardized 'airconditioning' to binary 1/0


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning
0,13300000,7420,4,2,3,1,0,0,0,1
1,12250000,8960,4,4,4,1,0,0,0,1
2,12250000,9960,3,2,2,1,0,1,0,0
3,12215000,7500,4,2,2,1,0,1,0,1
4,11410000,7420,4,1,2,1,1,1,0,1


## Step 6: Handling Outliers (IQR Method)
We check and filter outliers for the numerical columns `price` and `area` using the Interquartile Range method.

In [12]:
df_no_outliers = df_standardized.copy()
numeric_cols = ['price', 'area']

for col in numeric_cols:
    Q1 = df_no_outliers[col].quantile(0.25)
    Q3 = df_no_outliers[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_no_outliers[(df_no_outliers[col] < lower_bound) | (df_no_outliers[col] > upper_bound)]
    print(f"\nColumn '{col}':")
    print(f"  IQR Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Outliers detected: {outliers.shape[0]}")
    
    if outliers.shape[0] > 0:
        df_no_outliers = df_no_outliers[(df_no_outliers[col] >= lower_bound) & (df_no_outliers[col] <= upper_bound)]
        print(f"  Removed outliers. New shape: {df_no_outliers.shape}")

df_cleaned = df_no_outliers.reset_index(drop=True)


Column 'price':
  IQR Bounds: [-35000.00, 9205000.00]
  Outliers detected: 15
  Removed outliers. New shape: (530, 10)

Column 'area':
  IQR Bounds: [-604.88, 10468.12]
  Outliers detected: 13
  Removed outliers. New shape: (517, 10)


## Step 7: Saving the Cleaned Dataset
Save the cleaned dataset to `Housing-cleaned.csv` and show final metadata.

In [13]:
df_cleaned.to_csv('Housing-cleaned.csv', index=False)
print("Cleaned dataset saved successfully to 'Housing-cleaned.csv'!")
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_cleaned.shape}")
df_cleaned.head()

Cleaned dataset saved successfully to 'Housing-cleaned.csv'!
Original shape: (545, 10)
Cleaned shape: (517, 10)


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning
0,9100000,6000,4,1,2,1,0,1,0,0
1,9100000,6600,4,2,2,1,1,1,0,1
2,8960000,8500,3,2,4,1,0,0,0,1
3,8890000,4600,3,2,2,1,1,0,0,1
4,8855000,6420,3,2,2,1,0,0,0,1
